# Query 2: Aggregations per Payment Type
**Type:** Aggregation (SUM, AVG, COUNT, MAX, MIN)  
**Problem Statement:** Compute fare statistics (avg, sum, count, min, max) grouped by payment type. Payment codes: 1=Credit card, 2=Cash, 3=No charge, 4=Dispute, 5=Unknown, 6=Voided.

In [1]:
import time
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, IntegerType

spark = SparkSession.builder \
    .appName('Q2_Aggregation') \
    .master('local[*]') \
    .config('spark.sql.shuffle.partitions', '8') \
    .config('spark.driver.memory', '2g') \
    .config('spark.executor.memory', '2g') \
    .config('spark.eventLog.enabled', 'false') \
    .config('spark.ui.enabled', 'false') \
    .getOrCreate()

spark.sparkContext.setLogLevel('ERROR')
print('Spark version:', spark.version)

26/04/25 19:44:59 WARN Utils: Your hostname, mariam-VirtualBox resolves to a loopback address: 127.0.1.1; using 10.0.2.15 instead (on interface enp0s3)
26/04/25 19:44:59 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/25 19:45:00 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 3.5.1


In [2]:
df = spark.read.option('header', 'true').option('inferSchema', 'true') \
          .csv('../data/yellow_tripdata_2015-01.csv') \
          .sample(fraction=0.2, seed=42)

df = df.withColumnRenamed('tpep_pickup_datetime',  'pickup_datetime') \
       .withColumnRenamed('tpep_dropoff_datetime', 'dropoff_datetime') \
       .withColumnRenamed('fare_amount',            'fare') \
       .withColumnRenamed('passenger_count',        'passengers') \
       .withColumnRenamed('trip_distance',          'distance') \
       .withColumnRenamed('total_amount',           'total')

df = df.withColumn('fare',       F.col('fare').cast(DoubleType())) \
       .withColumn('total',      F.col('total').cast(DoubleType())) \
       .withColumn('distance',   F.col('distance').cast(DoubleType())) \
       .withColumn('passengers', F.col('passengers').cast(IntegerType())) \
       .withColumn('tip_amount', F.col('tip_amount').cast(DoubleType())) \
       .cache()

df.createOrReplaceTempView('trips')
rdd = df.rdd
print('Total rows (20% sample):', df.count())

Total rows (20% sample): 2233826


## RDD Implementation

In [3]:
start = time.time()

result_rdd = (
    rdd
    .filter(lambda r: r['payment_type'] is not None
                  and r['fare']         is not None)
    .map(lambda r: (r['payment_type'],
                    (r['fare'], r['fare'], r['fare'], r['fare'], 1)))
    .reduceByKey(lambda a, b: (
        a[0] + b[0],           # sum
        min(a[1], b[1]),       # min
        max(a[2], b[2]),       # max
        a[3] + b[3],           # sum for avg
        a[4] + b[4]            # count
    ))
    .mapValues(lambda x: {
        'sum':   round(x[0], 2),
        'min':   round(x[1], 2),
        'max':   round(x[2], 2),
        'avg':   round(x[3] / x[4], 2),
        'count': x[4]
    })
    .sortBy(lambda x: x[0])
    .collect()
)

rdd_time = time.time() - start
print(f'RDD | Time: {rdd_time:.2f}s')
print('Payment type stats (RDD):')
for pt, stats in result_rdd:
    print(f'  payment_type={pt}  {stats}')

RDD | Time: 18.15s
Payment type stats (RDD):
  payment_type=1.0  {'sum': 17253688.09, 'min': -52.0, 'max': 999.99, 'avg': 12.49, 'count': 1381058}
  payment_type=2.0  {'sum': 9226970.47, 'min': -200.0, 'max': 414.44, 'avg': 10.93, 'count': 844051}
  payment_type=3.0  {'sum': 70985.41, 'min': -100.0, 'max': 600.01, 'avg': 10.65, 'count': 6665}
  payment_type=4.0  {'sum': 21139.14, 'min': -242.25, 'max': 780.0, 'avg': 10.3, 'count': 2052}


## DataFrame Implementation

In [4]:
start = time.time()

result_df = (
    df.groupBy('payment_type')
      .agg(
          F.count('*')                        .alias('trip_count'),
          F.round(F.avg('fare'),   2)         .alias('avg_fare'),
          F.round(F.sum('fare'),   2)         .alias('total_fare'),
          F.round(F.min('fare'),   2)         .alias('min_fare'),
          F.round(F.max('fare'),   2)         .alias('max_fare')
      )
      .orderBy('payment_type')
)

print('--- Q2 DataFrame explain(True) ---')
result_df.explain(True)
result_df.cache()
df_time = time.time() - start
print(f'DataFrame | Time: {df_time:.2f}s')
result_df.show()
result_df.unpersist()

--- Q2 DataFrame explain(True) ---
== Parsed Logical Plan ==
'Sort ['payment_type ASC NULLS FIRST], true
+- Aggregate [payment_type#28], [payment_type#28, count(1) AS trip_count#1289L, round(avg(fare#176), 2) AS avg_fare#1291, round(sum(fare#176), 2) AS total_fare#1293, round(min(fare#176), 2) AS min_fare#1295, round(max(fare#176), 2) AS max_fare#1297]
   +- Project [VendorID#17, pickup_datetime#55, dropoff_datetime#76, passengers#236, distance#216, pickup_longitude#22, pickup_latitude#23, RateCodeID#24, store_and_fwd_flag#25, dropoff_longitude#26, dropoff_latitude#27, payment_type#28, fare#176, extra#30, mta_tax#31, cast(tip_amount#32 as double) AS tip_amount#256, tolls_amount#33, improvement_surcharge#34, total#196]
      +- Project [VendorID#17, pickup_datetime#55, dropoff_datetime#76, cast(passengers#116 as int) AS passengers#236, distance#216, pickup_longitude#22, pickup_latitude#23, RateCodeID#24, store_and_fwd_flag#25, dropoff_longitude#26, dropoff_latitude#27, payment_type#28, 

+------------+----------+--------+-------------+--------+--------+
|payment_type|trip_count|avg_fare|   total_fare|min_fare|max_fare|
+------------+----------+--------+-------------+--------+--------+
|         1.0|   1381058|   12.49|1.725368809E7|   -52.0|  999.99|
|         2.0|    844051|   10.93|   9226970.47|  -200.0|  414.44|
|         3.0|      6665|   10.65|     70985.41|  -100.0|  600.01|
|         4.0|      2052|    10.3|     21139.14| -242.25|   780.0|
+------------+----------+--------+-------------+--------+--------+



DataFrame[payment_type: double, trip_count: bigint, avg_fare: double, total_fare: double, min_fare: double, max_fare: double]

## Spark SQL Implementation

In [5]:
start = time.time()

result_sql = spark.sql("""
    SELECT   payment_type,
             COUNT(*)              AS trip_count,
             ROUND(AVG(fare), 2)   AS avg_fare,
             ROUND(SUM(fare), 2)   AS total_fare,
             ROUND(MIN(fare), 2)   AS min_fare,
             ROUND(MAX(fare), 2)   AS max_fare
    FROM     trips
    GROUP BY payment_type
    ORDER BY payment_type
""")

print('--- Q2 Spark SQL explain(True) ---')
result_sql.explain(True)
result_sql.cache()
sql_time = time.time() - start
print(f'SQL | Time: {sql_time:.2f}s')
result_sql.show()
result_sql.unpersist()

--- Q2 Spark SQL explain(True) ---
== Parsed Logical Plan ==
'Sort ['payment_type ASC NULLS FIRST], true
+- 'Aggregate ['payment_type], ['payment_type, 'COUNT(1) AS trip_count#2276, 'ROUND('AVG('fare), 2) AS avg_fare#2277, 'ROUND('SUM('fare), 2) AS total_fare#2278, 'ROUND('MIN('fare), 2) AS min_fare#2279, 'ROUND('MAX('fare), 2) AS max_fare#2280]
   +- 'UnresolvedRelation [trips], [], false

== Analyzed Logical Plan ==
payment_type: double, trip_count: bigint, avg_fare: double, total_fare: double, min_fare: double, max_fare: double
Sort [payment_type#28 ASC NULLS FIRST], true
+- Aggregate [payment_type#28], [payment_type#28, count(1) AS trip_count#2276L, round(avg(fare#176), 2) AS avg_fare#2277, round(sum(fare#176), 2) AS total_fare#2278, round(min(fare#176), 2) AS min_fare#2279, round(max(fare#176), 2) AS max_fare#2280]
   +- SubqueryAlias trips
      +- View (`trips`, [VendorID#17,pickup_datetime#55,dropoff_datetime#76,passengers#236,distance#216,pickup_longitude#22,pickup_latitude#23

+------------+----------+--------+-------------+--------+--------+
|payment_type|trip_count|avg_fare|   total_fare|min_fare|max_fare|
+------------+----------+--------+-------------+--------+--------+
|         1.0|   1381058|   12.49|1.725368809E7|   -52.0|  999.99|
|         2.0|    844051|   10.93|   9226970.47|  -200.0|  414.44|
|         3.0|      6665|   10.65|     70985.41|  -100.0|  600.01|
|         4.0|      2052|    10.3|     21139.14| -242.25|   780.0|
+------------+----------+--------+-------------+--------+--------+



DataFrame[payment_type: double, trip_count: bigint, avg_fare: double, total_fare: double, min_fare: double, max_fare: double]

## Performance Comparison

In [6]:
print('='*65)
row1 = f'{"Metric":<25} {"RDD":>12} {"DataFrame":>12} {"SQL":>12}'
row2 = f'{"Execution Time":<25} {rdd_time:>11.2f}s {df_time:>11.2f}s {sql_time:>11.2f}s'
row3 = f'{"Aggregation":<25} {"Manual":>12} {"Built-in":>12} {"Built-in":>12}'
row4 = f'{"Optimizer":<25} {"None":>12} {"Catalyst":>12} {"Catalyst":>12}'
print(row1)
print('-'*65)
print(row2)
print(row3)
print(row4)
print('='*65)
print()
print('KEY INSIGHT:')
print('RDD requires manual accumulator tuples for multi-metric aggregation.')
print('DataFrame/SQL use HashAggregate with Tungsten binary format.')
print('Catalyst generates optimized partial + final aggregation plans.')

Metric                             RDD    DataFrame          SQL
-----------------------------------------------------------------
Execution Time                  18.15s        0.25s        0.24s
Aggregation                     Manual     Built-in     Built-in
Optimizer                         None     Catalyst     Catalyst

KEY INSIGHT:
RDD requires manual accumulator tuples for multi-metric aggregation.
DataFrame/SQL use HashAggregate with Tungsten binary format.
Catalyst generates optimized partial + final aggregation plans.
